In [138]:
import win32com.client
import cv2
import photoshop.api as ps
from PIL import Image
import os
import pandas as pd
import time

In [192]:
def judge_type_mp3(path):
    video_capture = cv2.VideoCapture(path)
    frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if frame_width > frame_height:
        judge = "横屏"
    elif frame_width < frame_height:
        judge = "竖屏"
    else:
        judge = "正方形"
    # 释放视频捕获对象
    video_capture.release()
    return judge
def handle_text_box(text_layer):  
    print("正在更改文字图层的边框")
    left = text_layer.TextItem.Position[0]
    top = text_layer.TextItem.Position[1]
    width = text_layer.TextItem.Width
    height = text_layer.TextItem.Height
    new_height = calculate_new_height(text_layer)
    text_layer.TextItem.Height = new_height
def calculate_new_height(layer):
    text_content = layer.TextItem.Contents
    num_lines = int(len(text_content)/13) + 1
    line_height = layer.TextItem.leading  # 获取文本行高
    new_height = num_lines * line_height
    return new_height
def change_content(text_layer,content):
    print("正在更改文字图层中的具体内容")
    text_layer.TextItem.Contents = content
    change_word_space()
    handle_text_box(text_layer)
def move_layer(text_layer1,text_layer2):
    print("正在移动图层，为合并做准备")
    new_height = calculate_new_height(text_layer1)
    height = text_layer1.TextItem.Position[1] + new_height*(300/72) + 50
    left = text_layer2.TextItem.Position[0]
    text_layer2.TextItem.Position = (left,height)
def change_word_space():
    print("正在更改文字图层中的字距")
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    script_file_path ="C:\\Users\\86139\\Desktop\\人民日报数据处理\\执行修改字距动作.jsx"
    with open(script_file_path, "r", encoding="utf-8") as file:
        javascript_code = file.read()
    ps_app.DoJavaScript(javascript_code)
def merge_picture(judge):
    print("正在保存图片")
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    if judge == '竖屏':
        script_file_path ="C:\\Users\\86139\\Desktop\\人民日报数据处理\\保存图片竖屏.jsx"
    else:
        script_file_path ="C:\\Users\\86139\\Desktop\\人民日报数据处理\\保存图片横屏.jsx"
    with open(script_file_path, "r", encoding="utf-8") as file:
        javascript_code = file.read()
    ps_app.DoJavaScript(javascript_code)
    save_path = "C:\\Users\\86139\\Desktop\\人民日报数据处理\\out_put.png"
    img = Image.open(save_path)
    # 裁剪掉透明部分
    img = img.crop(img.getbbox())
    # 保存裁剪后的图片
    img.save(save_path)
def set_visible(judge):
    print("将要处理的组设置为可见")
    if judge == "竖屏": 
        n = 1
    else:
        n=2
    psApp = win32com.client.Dispatch("Photoshop.Application")
    doc = psApp.ActiveDocument
    group = doc.LayerSets.Item(n)
    group.Visible =  True  
def set_unvisible(judge):
    print("将处理完的组设置为不可见")
    if judge == "竖屏": 
        n = 1
    else:
        n=2
    psApp = win32com.client.Dispatch("Photoshop.Application")
    doc = psApp.ActiveDocument
    group = doc.LayerSets.Item(n)
    group.Visible =  False
def open_psd():
    print("正在打开ps，并等待25秒")
    # 连接到Photoshop
    app = win32com.client.Dispatch("Photoshop.Application")
    # 打开PSD文件
    psd_file_path = r"C:\Users\86139\Desktop\人民日报数据处理\文字处理.psd"
    doc = app.Open(psd_file_path)
    time.sleep(25)
def move_file(file):
    file_name = file.split('_')[1]
    print("正在移动图片")
    path = "C:\\Users\\86139\\Desktop\\人民日报数据"
    old_path = "C:\\Users\\86139\\Desktop\\人民日报数据处理\\out_put.png"
    new_path = os.path.join(path,file)
    new_path = os.path.join(new_path,file_name+'.png')
    os.rename(old_path,new_path)
    print("正在移动文件夹")
    old_path_file = os.path.join(path,file)
    path_save_file = "C:\\Users\\86139\\Desktop\\人民日报视频处理好的了"
    new_path_file = os.path.join(path_save_file,file)
    os.rename(old_path_file,new_path_file)
    print(file,"处理完毕")
def control_ps(file):
    file_name = file_name = file.split('_')[1]
    set_visible(dic_type[file_name])
    if dic_type[file_name] == '竖屏':
        n = 1
    else:
        n = 2
    psApp = win32com.client.Dispatch("Photoshop.Application")
    doc = psApp.ActiveDocument
    group = doc.LayerSets.Item(n)
    text_layer1 = group.ArtLayers.Item(2)
    text_layer2 = group.ArtLayers.Item(3)
    change_content(text_layer1,content=dic_content[file_name])
    move_layer(text_layer1,text_layer2)
    merge_picture(dic_type[file_name])
    move_file(file)
    set_unvisible(dic_type[file_name])
def quit_ps():
    print("正在等待5s")
    time.sleep(5)
    print("ps操作完成，关闭ps")
    ps_app = win32com.client.Dispatch("Photoshop.Application")
    doc = ps_app.ActiveDocument
    # 关闭文档并选择不保存
    doc.Close(2)  # 参数2表示不保存
    ps_app.Quit()
def handle_text_ps():
    open_psd()
    file_list = os.listdir("C:\\Users\\86139\\Desktop\\人民日报数据")
    data = pd.read_excel("C:\\Users\\86139\\Desktop\\人民日报数据处理\\人民日报数据的收集.xlsx")
    path = "C:\\Users\\86139\\Desktop\\人民日报数据"
    global dic_content
    global dic_type
    dic_content ={}
    dic_type = {}
    for file in file_list:
        path1 = os.path.join(path,file)
        file_name = file.split('_')[1]
        for i in range(data.shape[0]):
            if data.iloc[i,1] == file_name:
                dic_content[file_name] = data.iloc[i,2]
        inner_file_name = os.listdir(path1)[0]
        path2 = os.path.join(path1,inner_file_name)
        judge = judge_type_mp3(path2)
        dic_type[file_name] = judge 
    for file in file_list:
        print("正在处理",file)
        control_ps(file)
    quit_ps()

In [190]:
#将视频中的文字转换为图片
handle_text_ps()

正在打开ps，并等待25秒
正在处理 55_#打铁花遇上火龙火凤凰好壮观#
将要处理的组设置为可见
正在更改文字图层中的具体内容
正在更改文字图层中的字距
正在更改文字图层的边框
正在移动图层，为合并做准备
正在保存图片
正在移动图片
正在移动文件夹
55_#打铁花遇上火龙火凤凰好壮观# 处理完毕
将处理完的组设置为不可见
正在处理 56_神奇！#新疆一湖面如梦幻冰上森林#
将要处理的组设置为可见
正在更改文字图层中的具体内容
正在更改文字图层中的字距
正在更改文字图层的边框
正在移动图层，为合并做准备
正在保存图片
正在移动图片
正在移动文件夹
56_神奇！#新疆一湖面如梦幻冰上森林# 处理完毕
将处理完的组设置为不可见
正在处理 57_收藏！#一个动作帮助改善弯腰驼背#
将要处理的组设置为可见
正在更改文字图层中的具体内容
正在更改文字图层中的字距
正在更改文字图层的边框
正在移动图层，为合并做准备
正在保存图片
正在移动图片
正在移动文件夹
57_收藏！#一个动作帮助改善弯腰驼背# 处理完毕
将处理完的组设置为不可见
正在处理 58_#神秘人10年捐赠1000万元善款#
将要处理的组设置为可见
正在更改文字图层中的具体内容
正在更改文字图层中的字距
正在更改文字图层的边框
正在移动图层，为合并做准备
正在保存图片
正在移动图片
正在移动文件夹
58_#神秘人10年捐赠1000万元善款# 处理完毕
将处理完的组设置为不可见
